# Laboratorio 1 · Bitácora

**Nombre:**  
**Usuario de GitHub:**  
**Fecha:**  

---

> Los enunciados están en la guía del laboratorio. Aquí solo van tus
> predicciones, tus resultados y tus explicaciones.

> **La regla que hace que esto sirva de algo:** la predicción se escribe
> *antes* de ejecutar. Si la rellenas después ya sabiendo el resultado, el
> ejercicio no mide nada y tú no aprendes nada. Nadie va a comprobarlo:
> es un trato contigo mismo.


In [3]:
from rlrs.dp import greedy_policy, q_value, value_iteration
from rlrs.envs import ACTION_NAMES, GridWorld
from rlrs.evaluation import compare, evaluate
from rlrs.policies import GreedyTabularPolicy, RandomPolicy

# Este cuaderno es una bitácora: no define algoritmos, los usa.
# Si necesitas escribir una función que valga la pena conservar,
# va en src/rlrs/, no aquí.


## Mi variante

Ejecuta `uv run python scripts/variante.py` y anota lo que te tocó.


In [11]:
RUIDO  = 0.4   # <- rellena
COSTE  = -0.02   # <- rellena
GAMMA  = 0.9

mi_env = GridWorld(noise=RUIDO, step_reward=COSTE)
mi_env.noise, mi_env.step_reward


(0.4, -0.02)

---
## Ejercicio 1 · El respaldo a mano


**Antes de ejecutar.** ¿Cuál de las cuatro acciones crees que gana en (0,3), y por qué?

_Tu predicción:_ 


In [25]:
print(mi_env)           # o mi_env.render(), a ver cómo se dibuja
print(ACTION_NAMES)     # el orden de las acciones

[m for m in dir(mi_env) if not m.startswith('_')]

('arriba', 'derecha', 'abajo', 'izquierda')


['is_terminal',
 'is_wall',
 'max_steps',
 'n_actions',
 'n_cols',
 'n_rows',
 'n_states',
 'noise',
 'render_values',
 'reset',
 'reward_of',
 'start',
 'state_index',
 'state_pos',
 'step',
 'step_reward',
 'terminals',
 'transitions',
 'walls']

In [43]:
V, politica, barridos = value_iteration(mi_env, GAMMA)

estado = (0, 3)

for a in range(4):
    q = q_value(mi_env, V, estado, a, GAMMA)
    print(f"{ACTION_NAMES[a]:>10}: {q:.4f}")

mejor = max(range(4), key=lambda a: q_value(mi_env, V, estado, a, GAMMA))
print("\nGana:", ACTION_NAMES[mejor])
print("Barridos:", barridos)

    arriba: 0.7448
   derecha: 0.8284
     abajo: 0.5592
 izquierda: 0.5567

Gana: derecha
Barridos: 66


**Explica.** ¿Coincidió? ¿Por qué gana esa y no las otras?

_Tu respuesta:_ 

La diferencia de 0.084 entre derecha y arriba es el valor real de apuntar bien. Con ruido 0, esa diferencia sería enorme. Con ruido 0.4, apuntar al premio te compra ocho centésimas. Guarda ese número: en el Ejercicio 4 bajas el ruido a cero y vuelves a mirarlo. El contraste es el punto entero del laboratorio.


---
## Ejercicio 2 · Tu variante, medida


In [73]:
# tu código
print(mi_env.n_rows, "filas x", mi_env.n_cols, "columnas")
print("inicio:", mi_env.start)
print("terminales:", mi_env.terminals)
print("muros:", mi_env.walls)
print(mi_env.render_values(V))


4 filas x 5 columnas
inicio: (3, 0)
terminales: {(0, 4): 1.0, (1, 4): -1.0}
muros: {(2, 3), (1, 1)}
+0.37   +0.50   +0.63   +0.83    +1    
+0.28     ###   +0.50   +0.48    -1    
+0.21   +0.25   +0.36     ###   +0.05  
+0.16   +0.19   +0.24   +0.17   +0.10  


In [78]:
print("V(3,0):", V[mi_env.state_index((3, 0))])

pi = GreedyTabularPolicy(politica)
print(evaluate(mi_env, pi))

V(3,0): 0.157751366143185
avida          retorno +0.739 [+0.723, +0.755]  exito 100.0%  pasos  14.1


**Anota.** Barridos, V(3,0), retorno con su intervalo, tasa de éxito y pasos medios.

_Tus números:_ 

Barridos: 66
V(3,0): 0.1578
Retorno: +0.739, IC [+0.723, +0.755]
Éxito: 100.0%
Pasos medios: 14.1


---
## Ejercicio 3 · Subir gamma


**Antes de ejecutar.** Al pasar de 0,9 a 0,99: ¿qué le pasa al número de barridos? ¿Y a la política?

_Tu predicción:_ 


In [83]:
# tu código
V99, pol99, barridos99 = value_iteration(mi_env, 0.99)

print("Barridos:", barridos, "->", barridos99)
print("V(3,0):", V[mi_env.state_index((3,0))], "->", V99[mi_env.state_index((3,0))])
print("Casillas donde cambia la política:", (politica != pol99).sum())

for s in range(mi_env.n_states):
    if politica[s] != pol99[s]:
        print(mi_env.state_pos(s), ACTION_NAMES[politica[s]], "->", ACTION_NAMES[pol99[s]])

print(mi_env.render_values(V99))

Barridos: 66 -> 95
V(3,0): 0.157751366143185 -> 0.6424303247453194
Casillas donde cambia la política: 0
+0.77   +0.83   +0.88   +0.94    +1    
+0.72     ###   +0.83   +0.82    -1    
+0.68   +0.70   +0.76     ###   +0.54  
+0.64   +0.66   +0.69   +0.64   +0.59  


**Explica.** ¿Qué se movió mucho y qué se movió poco? ¿Por qué?

_Tu respuesta:_ 
Los valores se disparan, pero la politica no se ve afectada



---
## Ejercicio 4 · Quitar el ruido


**Antes de ejecutar.** Con ruido 0, ¿cambia la política óptima respecto a la tuya? ¿En qué casillas?

_Tu predicción:_ 


In [87]:
# tu código
env0 = GridWorld(noise=0.0, step_reward=COSTE)
V0, pol0, barridos0 = value_iteration(env0, GAMMA)

print("Barridos:", barridos, "->", barridos0)
print("V(3,0):", V[mi_env.state_index((3,0))], "->", V0[env0.state_index((3,0))])
print("Casillas donde cambia:", (politica != pol0).sum())

for s in range(env0.n_states):
    if politica[s] != pol0[s]:
        print(env0.state_pos(s), ACTION_NAMES[politica[s]], "->", ACTION_NAMES[pol0[s]])

print(env0.render_values(V0))

Barridos: 66 -> 9
V(3,0): 0.157751366143185 -> 0.43772920000000004
Casillas donde cambia: 1
(1, 3) izquierda -> arriba
+0.67   +0.77   +0.88   +1.00    +1    
+0.59     ###   +0.77   +0.88    -1    
+0.51   +0.59   +0.67     ###   +0.37  
+0.44   +0.51   +0.59   +0.51   +0.44  


In [93]:
pi0 = GreedyTabularPolicy(pol0)
for r in compare(env0, [pi0, pi]):
    print(r)

avida          retorno +0.880 [+0.880, +0.880]  exito 100.0%  pasos   7.0
avida          retorno +0.880 [+0.880, +0.880]  exito 100.0%  pasos   7.0


**Explica.** ¿Acertaste? Si te sorprendió, di exactamente qué esperabas y qué pasó.

_Tu respuesta:_ 


En el ejercicio se observó que, al eliminar el ruido, la política solo cambió en la casilla (1,3), ubicada cerca del estado terminal −1. En el mundo resbaladizo, el agente evita esa zona por el riesgo de caer accidentalmente, mientras que en el mundo determinista puede pasar directamente sin peligro. Además, el algoritmo necesitó menos barridos y el valor de la posición inicial aumentó, demostrando que las acciones confiables permiten obtener mejores resultados y rutas más eficientes.

---
## Ejercicio 5 · Encarecer el paso


**Antes de ejecutar.** Con coste por paso −2, ¿qué hará el agente?

_Tu predicción:_ 


In [96]:
# tu código
env2 = GridWorld(noise=RUIDO, step_reward=-2.0)
V2, pol2, barridos2 = value_iteration(env2, GAMMA)

print("Barridos:", barridos, "->", barridos2)
print("V(3,0):", V[mi_env.state_index((3,0))], "->", V2[env2.state_index((3,0))])
print("Casillas donde cambia:", (politica != pol2).sum())

for s in range(env2.n_states):
    if politica[s] != pol2[s]:
        print(env2.state_pos(s), ACTION_NAMES[politica[s]], "->", ACTION_NAMES[pol2[s]])

print(env2.render_values(V2))

Barridos: 66 -> 50
V(3,0): 0.157751366143185 -> -12.124226150740828
Casillas donde cambia: 9
(1, 2) arriba -> derecha
(1, 3) izquierda -> derecha
(2, 0) arriba -> derecha
(2, 4) abajo -> arriba
(3, 0) arriba -> derecha
(3, 1) arriba -> derecha
(3, 2) arriba -> derecha
(3, 3) izquierda -> derecha
(3, 4) izquierda -> arriba
-9.09   -6.50   -4.00   -0.65    +1    
-10.80     ###   -5.17   -1.85    -1    
-11.60   -10.13   -8.07     ###   -2.19  
-12.12   -10.84   -9.38   -7.85   -5.60  


In [98]:
pi2 = GreedyTabularPolicy(pol2)
print(evaluate(env2, pi2))

avida          retorno -17.780 [-18.658, -16.902]  exito 12.5%  pasos   9.5


**Explica.** ¿Qué está optimizando exactamente el agente para comportarse así?

_Tu respuesta:_ 


En este ejercicio se observó que, debido al alto costo de −2 por cada paso, el agente cambió su estrategia y comenzó a buscar el terminal más cercano, incluso si era el estado −1. Esto se debe a que llegar rápidamente al final resulta menos costoso que recorrer una mayor distancia para obtener el premio de +1. Los valores negativos, el bajo porcentaje de éxito del 12.5 % y la reducción en el número de pasos confirman que el agente está priorizando terminar el episodio lo antes posible.


---
## Ejercicio 6 · El error plantado


In [99]:
# ejecuta experiments/divergencia.py desde la terminal y pega aquí lo que salga


**Explica las dos capas del diagnóstico.**

_La capa matemática:_ 
Con y=1 y una recompensa positiva por cada paso, el agente puede obtener una recompensa infinita si nunca termina el episodio. Por esta razón, los valores aumentan constantemente y el algoritmo no converge.

_La capa de diseño:_ 
El problema no es únicamente usar y=1, sino la forma en que se definió la recompensa. Al premiar cada movimiento, se incentiva al agente a seguir moviéndose indefinidamente en lugar de alcanzar un objetivo final


---
## Cierre

**Lo que más me sorprendió hoy:** 

Lo más sorprendente fue que en el Ejercicio 6, aunque γ=1 fue rechazado inicialmente porque no garantiza la convergencia, el algoritmo sí logró converger correctamente cuando existía un costo negativo por cada paso. Esto demuestra que una condición matemática puede garantizar un resultado, pero su ausencia no significa necesariamente que el algoritmo vaya a fallar.

**Lo que todavía no entiendo:** 

Lo más difícil de entender fue por qué, al aumentar γ de 0.9 a 0.99 en el Ejercicio 3, los valores cambiaron considerablemente, pero la política permaneció exactamente igual. La dificultad está en comprender que los valores pueden aumentar porque se le da mayor importancia a las recompensas futuras, sin que esto necesariamente cambie cuál es la mejor acción en cada casilla.

